In [23]:
import networkx as nx
from shapely.ops import unary_union
from shapely.geometry import Polygon
from shapely.ops import polygonize
from skimage import measure
import numpy as np
import matplotlib.pyplot as plt
import cv2

from PIL import Image, ImageDraw

In [24]:
# polygon with a hole and bridge between shell and hole
shell = [(0, 0), (100, 0), (100, 100), (0, 100)]
hole = [(30, 30), (70, 30), (70, 70), (30, 70)]

polygon_points = [shell[0]] + [hole[0]] + hole[1:] + [hole[0]] + [shell[0]] + shell[1:]
polygon = Polygon(polygon_points)

In [25]:
def _draw_polygons(image, p, line_color, width=4, fill_color=None):
    # create a grayscale mask for the polygon
    mask = Image.new("L", image.size, 0)
    mask_draw = ImageDraw.Draw(mask)

    # draw outer polygon
    mask_draw.polygon(
        p.exterior.coords,
        fill=255,  # visible area
    )

    # cut out holes
    for hole in p.interiors:
        mask_draw.polygon(
            hole.coords,
            fill=0,  # invisible area
        )

    # apply the fill color using mask
    if fill_color is not None:
        r, g, b, a = fill_color
        base_layer = Image.new("RGBA", image.size, (r, g, b, 0))
        alpha_mask = mask.point(lambda p: int(p * (a / 255)))
        base_layer.putalpha(alpha_mask)
        image.alpha_composite(base_layer)

    # draw the border on top of the fill
    border_draw = ImageDraw.Draw(image, "RGBA")

    border_draw.polygon(
        p.exterior.coords,
        outline=line_color,
        width=width,
        fill=None,  # no fill for border, only outline
    )

    for hole in p.interiors:
        border_draw.polygon(
            hole.coords,
            outline=line_color,
            width=width,
            fill=None,  # no fill for border, only outline
        )
    return image

In [26]:
# create a blank RGBA image
image = Image.new("RGBA", (150,150), (255, 255, 255, 255))
drew_img = _draw_polygons(image, polygon, line_color=(255, 0, 0, 255), width=2, fill_color=(255, 0, 0, 128))

drew_img.show()